In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import heapq
import itertools
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, STEEL

In [ ]:
# hide
# autorun
N_MAX = 6                                     # longest block of tosses
P_GRID = [round(0.01 * k, 2) for k in range(1, 51)]   # the P(heads) slider's stops
N_START, P_START = 2, 0.1
LABEL_MAX = 8                                 # label the tree only up to this many leaves


def block_probs(n: int, p: float) -> dict[str, float]:
    """Every block of n tosses ("HTT", ...) and its probability."""
    return {"".join(b): p ** b.count("H") * (1 - p) ** b.count("T")
            for b in itertools.product("HT", repeat=n)}


def huffman(probs: dict[str, float]) -> str | tuple:
    """The Huffman tree: a leaf is an outcome, a node is a pair (bit-0 child, bit-1 child)."""
    # the counter breaks ties between equal probabilities, so the tree is deterministic
    heap = [(prob, k, outcome) for k, (outcome, prob) in enumerate(probs.items())]
    heapq.heapify(heap)
    k = len(heap)
    while len(heap) > 1:
        p_a, _, a = heapq.heappop(heap)       # the least probable node
        p_b, _, b = heapq.heappop(heap)       # the next least probable
        heapq.heappush(heap, (p_a + p_b, k, (b, a)))   # the more probable child gets bit 0
        k += 1
    return heap[0][2]


def codewords(tree: str | tuple, prefix: str = "") -> dict[str, str]:
    """Read each outcome's codeword off the tree: the bits on the path from the root."""
    if isinstance(tree, str):
        return {tree: prefix}
    return {**codewords(tree[0], prefix + "0"), **codewords(tree[1], prefix + "1")}


def bits_per_toss(n: int, p: float) -> float:
    """Expected codeword length per block, divided by the n tosses in a block."""
    probs = block_probs(n, p)
    codes = codewords(huffman(probs))
    return sum(probs[b] * len(codes[b]) for b in probs) / n


def entropy(p: float) -> float:
    return -p * np.log2(p) - (1 - p) * np.log2(1 - p) if 0 < p < 1 else 0.0


def tree_layout(tree: str | tuple) -> dict[str, list]:
    """Plot coordinates: leaves left to right in order, each node above its children, y = depth."""
    lay = {"edge_x": [], "edge_y": [], "node_x": [], "node_y": [],
           "leaf_x": [], "leaf_y": [], "leaves": [], "bit_x": [], "bit_y": [], "bits": []}
    count = itertools.count()

    def place(node, depth):
        if isinstance(node, str):
            x = next(count)
            lay["leaf_x"].append(x)
            lay["leaf_y"].append(depth)
            lay["leaves"].append(node)
            return x
        xs = [place(child, depth + 1) for child in node]
        x = (xs[0] + xs[1]) / 2
        lay["node_x"].append(x)
        lay["node_y"].append(depth)
        for bit, x_child in enumerate(xs):
            lay["edge_x"] += [x, x_child, None]
            lay["edge_y"] += [depth, depth + 1, None]
            lay["bit_x"].append((x + x_child) / 2)
            lay["bit_y"].append(depth + 0.5)
            lay["bits"].append(str(bit))
        return x

    place(tree, 0)
    return lay


# the right panel's curves and each N's deepest tree (so the axes only move when N does)
BITS = {p: [bits_per_toss(n, p) for n in range(1, N_MAX + 1)] for p in P_GRID}
DEPTH = {n: max(len(c) for p in P_GRID for c in codewords(huffman(block_probs(n, p))).values())
         for n in range(1, N_MAX + 1)}


def tree_traces(n: int, p: float) -> dict:
    """Everything the left panel draws for one (N, p)."""
    probs = block_probs(n, p)
    tree = huffman(probs)
    codes = codewords(tree)
    lay = tree_layout(tree)
    labeled = len(probs) <= LABEL_MAX
    lay["labels"] = [f"{b}<br>{codes[b]}" if labeled else "" for b in lay["leaves"]]
    lay["hover"] = [f"{b}: P = {probs[b]:.3g}, codeword {codes[b]}" for b in lay["leaves"]]
    if not labeled:
        lay["bit_x"], lay["bit_y"], lay["bits"] = [], [], []
    lay["bit_pos"] = ["top left" if b == "0" else "top right" for b in lay["bits"]]
    lay["leaf_size"] = 8 if len(probs) <= 16 else 5
    lay["x_range"] = [-0.7, len(probs) - 0.3]
    lay["y_range"] = [DEPTH[n] + (1.1 if labeled else 0.5), -0.5]
    lay["y_dtick"] = 1 if DEPTH[n] <= 6 else None   # None: plotly picks the ticks
    return lay


def readout(n: int, p: float) -> str:
    per_toss = BITS[p][n - 1]
    h = entropy(p)
    return (f"N = {n}: {per_toss * n:.2f} bits per block → {per_toss:.3f} bits per toss "
            f"({1 / per_toss:.2f}× compression). "
            f"Entropy (theoretical limit): {h:.3f} bits per toss ({1 / h:.2f}×).")


def figure():
    fig = make_subplots(rows=1, cols=2, column_widths=[0.62, 0.38], horizontal_spacing=0.1)
    lay = tree_traces(N_START, P_START)
    # left: the tree (0 edges, 1 internal nodes, 2 leaves, 3 edge bits)
    fig.add_scatter(x=lay["edge_x"], y=lay["edge_y"], mode="lines",
                    line=dict(color=IRON, width=1.2), hoverinfo="skip", row=1, col=1)
    fig.add_scatter(x=lay["node_x"], y=lay["node_y"], mode="markers",
                    marker=dict(color=IRON, size=5), hoverinfo="skip", row=1, col=1)
    fig.add_scatter(x=lay["leaf_x"], y=lay["leaf_y"], mode="markers+text",
                    marker=dict(color=RED, size=lay["leaf_size"]), text=lay["labels"],
                    textposition="bottom center", textfont=dict(size=11),
                    hovertext=lay["hover"], hoverinfo="text", row=1, col=1)
    fig.add_scatter(x=lay["bit_x"], y=lay["bit_y"], mode="text", text=lay["bits"],
                    textposition=lay["bit_pos"], textfont=dict(size=11, color=IRON),
                    hoverinfo="skip", row=1, col=1)
    fig.update_xaxes(range=lay["x_range"], showline=False, ticks="", showticklabels=False,
                     fixedrange=True, row=1, col=1)
    fig.update_yaxes(range=lay["y_range"], dtick=lay["y_dtick"],
                     title_text="Codeword length (bits)", fixedrange=True, row=1, col=1)

    # right: bits per toss against block length (4 entropy, 5 Huffman, 6 the current N)
    ns = list(range(1, N_MAX + 1))
    h = entropy(P_START)
    fig.add_scatter(x=[0.5, 0.65, N_MAX + 0.5], y=[h, h, h], mode="lines+text",
                    line=dict(color=RED, width=1.6, dash="dash"), text=["", "entropy (theoretical limit)", ""],
                    textposition="bottom right", textfont=dict(color=RED),
                    hovertemplate="entropy (theoretical limit): %{y:.3f} bits per toss<extra></extra>",
                    row=1, col=2)
    fig.add_scatter(x=ns, y=BITS[P_START], mode="lines+markers",
                    line=dict(color=BLUE, width=2), marker=dict(color=BLUE, size=7),
                    hovertemplate="N = %{x}: %{y:.3f} bits per toss<extra>Huffman</extra>",
                    row=1, col=2)
    fig.add_scatter(x=[N_START], y=[BITS[P_START][N_START - 1]], mode="markers",
                    marker=dict(color=GOLD, size=13, line=dict(color=IRON, width=1)),
                    hoverinfo="skip", row=1, col=2)
    fig.update_xaxes(range=[0.5, N_MAX + 0.5], dtick=1, title_text="Block length N",
                     fixedrange=True, row=1, col=2)
    fig.update_yaxes(range=[0, 1.05], title_text="Bits per toss", fixedrange=True,
                     row=1, col=2)
    fig.update_layout(height=420)
    return fig


def controls(fig):
    n_slider = widgets.IntSlider(description="Block length $N$", min=1, max=N_MAX,
                                 value=N_START)
    p_slider = widgets.FloatSlider(description="P(heads) $p$", min=0.01, max=0.5, step=0.01,
                                   value=P_START, readout_format=".2f")
    text = widgets.HTML(f"<span style='font-size:0.9em'>{readout(N_START, P_START)}</span>")

    # the defaults snapshot the helpers; the page's notebooks share one kernel
    def update(n, p, tree_traces=tree_traces, readout=readout, entropy=entropy, BITS=BITS):
        p = round(p, 2)
        lay = tree_traces(n, p)
        h = entropy(p)
        with fig.batch_update():
            fig.data[0].x, fig.data[0].y = lay["edge_x"], lay["edge_y"]
            fig.data[1].x, fig.data[1].y = lay["node_x"], lay["node_y"]
            fig.data[2].x, fig.data[2].y = lay["leaf_x"], lay["leaf_y"]
            fig.data[2].text, fig.data[2].hovertext = lay["labels"], lay["hover"]
            fig.data[2].marker.size = lay["leaf_size"]
            fig.data[3].x, fig.data[3].y = lay["bit_x"], lay["bit_y"]
            fig.data[3].text, fig.data[3].textposition = lay["bits"], lay["bit_pos"]
            fig.layout.xaxis.range = lay["x_range"]
            fig.layout.yaxis.range = lay["y_range"]
            fig.layout.yaxis.dtick = lay["y_dtick"]
            fig.data[4].y = [h, h, h]
            fig.data[5].y = BITS[p]
            fig.data[6].x, fig.data[6].y = [n], [BITS[p][n - 1]]
        text.value = f"<span style='font-size:0.9em'>{readout(n, p)}</span>"

    widgets.interactive_output(update, {"n": n_slider, "p": p_slider})
    return widgets.VBox([n_slider, p_slider, text])


icm_plotly.show(figure, controls)